In [1]:
import matplotlib.pyplot as plt
import pickle

from pmbrl.model2 import Model
from pmbrl.data import Experiment_Data, get_data_expanded

In [2]:
nome_do_arquivo = 'regular_normilized.pkl'

with open(nome_do_arquivo, 'rb') as arquivo:
    exp = pickle.load(arquivo)
    data = exp['data']
    model = exp['model']

del exp
del arquivo

# Calculating AVG

In [3]:
expansions = {
    'estimated_p': ['p0', 'p1'],
    's_': ['s0', 's1', 's2', 's3'],
    's__': ['s_0', 's_1', 's_2', 's_3'],
}

df = get_data_expanded(data.evaluation_data, expansions)
df_avgs = df.groupby('episode')[['p0', 'p1']].mean().reset_index()

def get_avg_per_epi(row):
    return df_avgs.loc[df_avgs['episode'] == row.episode][['p0', 'p1']].values[0]
df[['avg_p0', 'avg_p1']] = df.apply(get_avg_per_epi, axis=1, result_type='expand')

df[['episode', 'avg_p0', 'avg_p1']]

,episode,avg_p0,avg_p1
0,0,0.263167,-0.249500
1,0,0.263167,-0.249500
2,0,0.263167,-0.249500
3,0,0.263167,-0.249500
4,0,0.263167,-0.249500
...,...,...,...
2122,99,0.401200,-0.616733
2123,99,0.401200,-0.616733
2124,99,0.401200,-0.616733
2125,99,0.401200,-0.616733


In [4]:
df_std = df.groupby('episode')[['p0', 'p1']].std().reset_index()
df_min = df.groupby('episode')[['p0', 'p1']].min().reset_index()
df_max = df.groupby('episode')[['p0', 'p1']].max().reset_index()
df_count = df.groupby('episode')[['p0', 'p1']].count().reset_index()
df_stats = df_avgs[['episode']].copy()

df_stats[['avg_p0', 'avg_p1']] = df_avgs[['p0', 'p1']]
df_stats[['std_p0', 'std_p1']] = df_std[['p0', 'p1']]
df_stats[['min_p0', 'min_p1']] = df_min[['p0', 'p1']]
df_stats[['max_p0', 'max_p1']] = df_max[['p0', 'p1']]
df_stats['range_p0'] = df_stats['max_p0'] - df_stats['min_p0']
df_stats['range_p1'] = df_stats['max_p1'] - df_stats['min_p1']
df_stats[['count']] = df_count[['p0']]

df_stats

,episode,avg_p0,avg_p1,std_p0,std_p1,min_p0,min_p1,max_p0,max_p1,range_p0,range_p1,count
0,0,0.263167,-0.249500,0.107747,0.096390,0.126,-0.376,0.394,-0.107,0.268,0.269,6
1,1,0.137444,-0.426889,0.053870,0.024028,0.022,-0.457,0.188,-0.391,0.166,0.066,9
2,2,0.403500,-0.460437,0.049810,0.115707,0.309,-0.638,0.464,-0.290,0.155,0.348,16
3,3,0.381053,-0.537053,0.089796,0.058478,0.275,-0.640,0.604,-0.422,0.329,0.218,19
4,4,0.303615,-0.508538,0.069197,0.014408,0.174,-0.532,0.398,-0.478,0.224,0.054,13
...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,0.235800,-0.431800,0.079231,0.096530,0.122,-0.577,0.365,-0.219,0.243,0.358,15
96,96,0.242605,-0.468684,0.072447,0.082550,0.071,-0.623,0.332,-0.285,0.261,0.338,38
97,97,0.285829,-0.577943,0.104485,0.098962,0.182,-0.756,0.634,-0.438,0.452,0.318,35
98,98,0.227060,-0.518566,0.053149,0.209695,0.131,-0.880,0.390,-0.072,0.259,0.808,83


In [5]:
df_stats.describe()

,episode,avg_p0,avg_p1,std_p0,std_p1,min_p0,min_p1,max_p0,max_p1,range_p0,range_p1,count
count,100.000000,100.000000,100.000000,100.000000,100.000000,100.00000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
mean,49.500000,0.287408,-0.527293,0.090436,0.090630,0.15345,-0.677100,0.443100,-0.374320,0.289650,0.302780,21.270000
std,29.011492,0.097561,0.127969,0.037165,0.047504,0.11355,0.153773,0.149607,0.151881,0.110103,0.156101,14.489882
min,0.000000,0.097636,-1.001000,0.019935,0.014408,-0.06300,-1.173000,0.188000,-0.832000,0.061000,0.050000,3.000000
25%,24.750000,0.222412,-0.588185,0.069989,0.055630,0.04450,-0.771750,0.313750,-0.476500,0.230250,0.181500,11.000000
50%,49.500000,0.294053,-0.516625,0.084290,0.085369,0.16550,-0.639500,0.463000,-0.391000,0.278500,0.289000,16.500000
75%,74.250000,0.360067,-0.443646,0.108383,0.113422,0.23975,-0.574000,0.537250,-0.259000,0.333250,0.363250,29.250000
max,99.000000,0.539500,-0.249500,0.239084,0.284304,0.38600,-0.376000,0.754000,-0.005000,0.676000,0.808000,83.000000


# Infering with avg

In [6]:
import torch
import torch.nn as nn

m = model.transition_estimator.state_layer

input_values = torch.tensor(df[['s0', 's1', 's2', 's3', 'a_', 'avg_p0', 'avg_p1']].values) 
target_value = torch.tensor(df[['s_0', 's_1', 's_2', 's_3']].values)

for p in m.parameters():
    p.requires_grad = False

output = m(input_values.float()) 


def normilize(v): 
        mins = input_values[:,0:4].min(axis=0).values.repeat((v.shape[0], 1))
        maxs = input_values[:,0:4].max(axis=0).values.repeat((v.shape[0], 1))
        rang = maxs - mins
        return (v - mins) / rang
loss = nn.MSELoss(reduction='none')(normilize(output).float(), normilize(target_value).float())
# loss = nn.MSELoss(reduction='none')(normilize(output).float(), (target_value).float())
loss = torch.sqrt(loss.mean(axis=0).sum())
loss

tensor(0.0902)

In [7]:
results = data.get_evaluation_metrics()
results.rse.mean()

np.float64(0.1960789844851904)

# Optimazing p

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim

history = []

for epi, p1, p2 in df_avgs.values:
    print(f'{epi:}')
    d = df[df['episode'] == epi].copy().reset_index(drop=True)

    target_value = torch.tensor(d[['s_0', 's_1', 's_2', 's_3']].values)
    param = torch.tensor([p1, p2], requires_grad=True)
    print(f'initial value for input Param: {[round(p,4) for p in param.tolist()]}') 
    input_values = torch.tensor(d[['s0', 's1', 's2', 's3', 'a_']].values) 

    learning_rate = 0.1
    num_epochs = 500

    optimizer = optim.Adam([param], lr=learning_rate)
    criterion = nn.MSELoss(reduction='none')

    for p in m.parameters():
        p.requires_grad = False

    for epoch in range(num_epochs):
        # Forward pass
        state_inputs = torch.concat([input_values, param.repeat((input_values.shape[0], 1))], dim=1)
        output = m(state_inputs.float())  # Add batch dimension

        # Calculate the loss
        # loss = criterion(output.float(), target_value.float())
        
        def normilize(v): 
            mins = input_values[:,:-1].min(axis=0).values.repeat((v.shape[0], 1))
            maxs = input_values[:,:-1].max(axis=0).values.repeat((v.shape[0], 1))
            rang = maxs - mins
            return (v - mins) / rang
        open_loss = criterion(normilize(output).float(), normilize(target_value).float())
        loss = torch.sqrt(open_loss.sum(axis=1).mean())
        # loss = torch.sqrt(open_loss.mean(axis=1).sum())
        # loss = open_loss.mean()
        # loss = open_loss.sum()

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # history.append((epoch, param.tolist(), loss.item()))

        # if (epoch + 1) % 100 == 0:
        #     # print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Input Param: {param.item():.4f}, Output: {output.item():.4f}')
        #     print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Input Param: {[round(p,4) for p in param.tolist()]}')
    history.append({'episode': epi, 'optimized_p': param.tolist(), 'loss':  loss.item()})
    print(f'Loss: {loss.item():.4f}, Input Param: {[round(p,4) for p in param.tolist()]}')


0.0
initial value for input Param: [0.2632, -0.2495]
Loss: 0.9581, Input Param: [0.1882, -0.3046]
1.0
initial value for input Param: [0.1374, -0.4269]
Loss: 0.1657, Input Param: [0.1448, -0.4258]
2.0
initial value for input Param: [0.4035, -0.4604]
Loss: 0.3885, Input Param: [0.4105, -0.4616]
3.0
initial value for input Param: [0.3811, -0.5371]
Loss: 0.2390, Input Param: [0.3886, -0.5344]
4.0
initial value for input Param: [0.3036, -0.5085]
Loss: 0.0997, Input Param: [0.3048, -0.5082]
5.0
initial value for input Param: [0.1994, -0.3986]
Loss: 0.2655, Input Param: [0.2, -0.3998]
6.0
initial value for input Param: [0.1254, -0.4731]
Loss: 0.3943, Input Param: [0.1188, -0.4757]
7.0
initial value for input Param: [0.3026, -0.5477]
Loss: 0.1866, Input Param: [0.3132, -0.5556]
8.0
initial value for input Param: [0.2667, -0.3101]
Loss: 1.8442, Input Param: [-0.0171, -0.4724]
9.0
initial value for input Param: [0.2349, -0.5643]
Loss: 0.2666, Input Param: [0.2273, -0.5564]
10.0
initial value for

In [9]:
import pandas as pd
df_optim = pd.DataFrame(history)


expansions = {
    'optimized_p': ['opt_p0', 'opt_p1'],
}

df_optim = get_data_expanded(df_optim, expansions)[['episode', 'loss', 'opt_p0', 'opt_p1']]
df_optim.loss.mean()

np.float64(0.47051977224648)

In [10]:
import torch
import torch.nn as nn

def get_opt_per_epi(row):
    return df_optim.loc[df_optim['episode'] == row.episode][['opt_p0', 'opt_p1']].values[0]
df[['opt_p0', 'opt_p1']] = df.apply(get_opt_per_epi, axis=1, result_type='expand')


input_values = torch.tensor(df[['s0', 's1', 's2', 's3', 'a_', 'opt_p0', 'opt_p1']].values) 
target_value = torch.tensor(df[['s_0', 's_1', 's_2', 's_3']].values)

for p in m.parameters():
    p.requires_grad = False

output = m(input_values.float()) 


def normilize(v): 
        mins = input_values[:,0:4].min(axis=0).values.repeat((v.shape[0], 1))
        maxs = input_values[:,0:4].max(axis=0).values.repeat((v.shape[0], 1))
        rang = maxs - mins
        return (v - mins) / rang
loss = nn.MSELoss(reduction='none')(normilize(output).float(), normilize(target_value).float())
# loss = nn.MSELoss(reduction='none')(normilize(output).float(), (target_value).float())
loss = torch.sqrt(loss.mean(axis=0).sum())
loss

tensor(0.1102)

In [11]:
del model
del data